# 11 · Practical F — Constructing a Volatility Smile

**Read first:** Chapter 12 (*Volatility Smile Market Instruments and Exposures*), then Practical F.

This notebook ends with the assembled volatility surface — Practicals D, E and F joined together, which the book never does.

---

## What you'll be able to do after this

- Build a volatility smile from the three instruments a desk actually quotes, and say what each one does to the shape.
- Convert between strike and delta in both directions, with the sign convention right.
- Assemble a full surface and state, precisely, what it is ignoring.

## The intuition, before the maths

### The problem the smile solves

Black-Scholes assumes one volatility. The market quotes a different one for every strike. Not because anyone is confused — because the model's assumption is wrong, and the market prices the ways it is wrong.

So a desk needs a rule: *given a strike, what volatility?* The smile is that rule.

### Three numbers, three jobs

The interbank market does not quote a curve. It quotes **three numbers per tenor**, and each does one job:

| Instrument | What it controls | Picture |
|---|---|---|
| **ATM** | the level | moves the whole smile up or down |
| **Risk reversal** | the tilt | rotates it — one side up, the other down |
| **Butterfly** | the wings | lifts both edges symmetrically |

Level, tilt, curvature. Three degrees of freedom, and a quadratic has exactly three. That is the Malz formula, and once you see it that way there is nothing mysterious left in it.

### Why *these* three?

Chapter 12's real answer is about **exposures**, not geometry. Each instrument isolates one greek at inception:

- An **ATM** contract has vega and nothing else — no vanna, no volga.
- A **risk reversal** has zero vega and pure **vanna** (∂vega/∂spot).
- A **butterfly** has zero vega, zero vanna and pure **volga** (∂vega/∂vol).

So they are not three arbitrary points on a curve. They are three clean instruments, each trading one distinct risk: the *level* of volatility, the *spot-versus-volatility relationship*, and the *volatility of volatility*.

### Why quote in delta rather than strike?

Because a strike is meaningless without context. "1.2500" tells you nothing until you know spot. "25 delta put" is self-locating — it means roughly a one-in-four chance of finishing in the money, at any spot, in any pair, at any tenor.

The market quotes in delta space and derives strikes from it. That ordering matters and it is why Practical F spends most of its length on converting between the two.

## The maths, derived not asserted

### The standard approximations

For the 25 delta strikes:

$$\sigma_{25dC} = \sigma_{ATM} + \sigma_{fly} + \tfrac{1}{2}\sigma_{RR}, \qquad \sigma_{25dP} = \sigma_{ATM} + \sigma_{fly} - \tfrac{1}{2}\sigma_{RR}$$

Invert them and you recover the definitions:

$$\sigma_{RR} = \sigma_{25dC} - \sigma_{25dP}, \qquad \sigma_{fly} = \tfrac{\sigma_{25dC} + \sigma_{25dP}}{2} - \sigma_{ATM}$$

The risk reversal is the **difference** between the two wings (tilt). The butterfly is their **average minus the ATM** (how far the wings sit above the middle).

### Malz (1997)

Allan Malz generalised those to any delta. With $X$ the **positive quoted put delta**:

$$\sigma(X) = \sigma_{ATM} + 2\,\sigma_{RR25}(X - 0.5) + 16\,\sigma_{fly25}(X - 0.5)^2$$

Check the coefficients rather than accepting them:

- At $X = 0.5$ both correction terms vanish → $\sigma_{ATM}$. ✓
- At $X = 0.25$: $(X - 0.5) = -0.25$, so the RR term is $2 \times (-0.25) = -0.5$ times $\sigma_{RR}$, and the fly term is $16 \times 0.0625 = 1$ times $\sigma_{fly}$ → $\sigma_{ATM} + \sigma_{fly} - \tfrac{1}{2}\sigma_{RR}$. ✓

The 2 and the 16 are not fitted parameters. They are exactly what makes the quadratic pass through the quoted points.

**Known limitation.** Substitute 10% and 90% and you get $\sigma_{RR10} = 1.6\,\sigma_{RR25}$. Chapter 12 says the market value is usually nearer **1.8**, so this form systematically understates the 10 delta skew. A property of the parameterisation, not a bug.

### Strike from delta

$$\Delta_{put} = e^{-r_1 T}\left[N(d_1) - 1\right]$$

Invert:

$$K = \frac{S}{\exp\left(N^{-1}\!\left(e^{r_1 T}\Delta_{put} + 1\right)\sigma\sqrt{T} - \left(r_2 - r_1 + \tfrac{\sigma^2}{2}\right)T\right)}$$

**The delta here is the true, negative value.** Pass $-0.25$ for a 25 delta put. The market quotes it positive; every formula wants it signed. The book flags this twice, which tells you how often it goes wrong.

A consequence worth noticing, which the book's zero-rate examples never surface: since $N(d_1) - 1 \in (-1, 0)$, the put delta is bounded by $-e^{-r_1 T}$. **With a 10% CCY1 rate over a year there is no such thing as a 95 delta put.** Experiment 5.

## The code

In [1]:
from datetime import date
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from fxds.smile import (
    MalzSmile, strike_placement, smile_by_strike,
    put_delta_from_strike, strike_from_put_delta,
    max_attainable_put_delta, STANDARD_PUT_DELTAS,
)
from fxds.surface import VolatilitySurface, TenorSmile, example_surface, SIMPLIFICATIONS
from fxds.blackscholes import forward
from fxds.plotting import (
    use_house_style, style_axis, mark_level, as_percent, style_plotly,
    PRIMARY, SECONDARY, TERTIARY, QUATERNARY, MUTED, ALERT,
)

use_house_style()
pd.set_option("display.precision", 6)

# A EUR/USD-shaped 1yr smile: downside rich, wings up.
SMILE = MalzSmile(atm=0.10, rr25=-0.02, fly25=0.005)
SPOT, T, R1, R2 = 1.30, 1.0, 0.02, 0.05

### Task A — verify the formula against the standard approximations

In [2]:
checks = [
    ("50% delta (ATM)", 0.50, SMILE.atm),
    ("25 delta put",    0.25, SMILE.put_25d),
    ("25 delta call",   0.75, SMILE.call_25d),
    ("10 delta put",    0.10, SMILE.put_10d),
    ("10 delta call",   0.90, SMILE.call_10d),
]
print(f"{'strike':<18} {'Malz formula':>14} {'approximation':>15}  match")
for label, x, approx in checks:
    v = SMILE.volatility(x)
    print(f"{label:<18} {v:>14.4%} {approx:>15.4%}  {np.isclose(v, approx)}")

print(f"\nrecovered RR25  = {SMILE.call_25d - SMILE.put_25d:+.4%}  (input {SMILE.rr25:+.4%})")
print(f"recovered fly25 = {(SMILE.call_25d + SMILE.put_25d)/2 - SMILE.atm:+.4%}  "
      f"(input {SMILE.fly25:+.4%})")
print(f"\nimplied RR10 = {SMILE.rr10_implied:+.4%} = 1.6 x RR25")
print(f"  Ch. 12 notes the market multiplier is usually nearer 1.8, so Malz")
print(f"  understates 10 delta skew. A known limitation of the functional form.")

strike               Malz formula   approximation  match
50% delta (ATM)          10.0000%        10.0000%  True
25 delta put             11.5000%        11.5000%  True
25 delta call             9.5000%         9.5000%  True
10 delta put             12.8800%        12.8800%  True
10 delta call             9.6800%         9.6800%  True

recovered RR25  = -2.0000%  (input -2.0000%)
recovered fly25 = +0.5000%  (input +0.5000%)

implied RR10 = -3.2000% = 1.6 x RR25
  Ch. 12 notes the market multiplier is usually nearer 1.8, so Malz
  understates 10 delta skew. A known limitation of the functional form.


### Task B — what each instrument does to the shape

Fixed y-axis, as the book insists — so the *shape* changes rather than the axis rescaling.

In [3]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), sharey=True)
deltas = np.linspace(0.01, 0.99, 200)

for value, colour in zip([0.08, 0.10, 0.12], [TERTIARY, PRIMARY, SECONDARY]):
    axes[0].plot(deltas, MalzSmile(atm=value, rr25=-0.02, fly25=0.005).volatility(deltas),
                 color=colour, label=f"ATM {value:.0%}")
axes[0].legend(fontsize=9)
style_axis(axes[0], "ATM moves the level", "Put delta", "Implied volatility",
           "Parallel shift. The shape is untouched.")

for value, colour in zip([-0.04, 0.0, 0.04], [TERTIARY, MUTED, SECONDARY]):
    axes[1].plot(deltas, MalzSmile(atm=0.10, rr25=value, fly25=0.005).volatility(deltas),
                 color=colour, label=f"RR {value:+.0%}")
axes[1].legend(fontsize=9)
style_axis(axes[1], "Risk reversal tilts it", "Put delta", "",
           "Rotates about the ATM. One wing up, the other down - the ATM never moves.")

for value, colour in zip([0.0, 0.005, 0.015], [MUTED, PRIMARY, SECONDARY]):
    axes[2].plot(deltas, MalzSmile(atm=0.10, rr25=-0.02, fly25=value).volatility(deltas),
                 color=colour, label=f"fly {value:.1%}")
axes[2].legend(fontsize=9)
style_axis(axes[2], "Butterfly lifts the wings", "Put delta", "",
           "Symmetric lift at both edges. The ATM stays put again.")

for ax in axes:
    ax.set_ylim(0.06, 0.16)
    as_percent(ax, decimals=0)
    mark_level(ax, 0.5, "ATM")
plt.tight_layout(); plt.show()

Three orthogonal controls. Notice the ATM is fixed under both the risk reversal and the butterfly — that is deliberate, and it is why a desk can move one without disturbing the others.

### Task C — strike from delta, and the round trip

The book's test: put a strike in, get a delta, put the delta back, get the strike out. If they match as the other inputs change, the formulas are right.

In [4]:
print(f"{'strike in':>10} {'put delta':>12} {'strike out':>12} {'error':>10}")
for strike in [1.10, 1.20, 1.30, 1.40, 1.55]:
    d = put_delta_from_strike(SPOT, strike, T, R1, R2, 0.10)
    back = strike_from_put_delta(SPOT, d, T, R1, R2, 0.10)
    print(f"{strike:>10.4f} {d:>12.6f} {back:>12.6f} {abs(back-strike):>10.2e}")

print("\nExact to machine precision. Note every delta is NEGATIVE - that is the")
print("true value. The market would quote these as positive numbers.")

 strike in    put delta   strike out      error
    1.1000    -0.021235     1.100000   0.00e+00
    1.2000    -0.122509     1.200000   0.00e+00
    1.3000    -0.355978     1.300000   0.00e+00
    1.4000    -0.639218     1.400000   0.00e+00
    1.5500    -0.902340     1.550000   2.22e-16

Exact to machine precision. Note every delta is NEGATIVE - that is the
true value. The market would quote these as positive numbers.


> **The trap.** Pass the positive quoted delta by mistake and you get a strike on the wrong side of the forward — silently, with no error, and it looks plausible. This implementation rejects it:

In [5]:
try:
    strike_from_put_delta(SPOT, 0.25, T, R1, R2, 0.10)
except ValueError as exc:
    print(f"ValueError: {exc}")

ValueError: put_delta must be the signed value strictly between -1 and 0, got 0.25. For a 25 delta put pass -0.25, not 0.25.


### Task D — the smile in both spaces

In [6]:
by_delta = SMILE.curve()
by_strike = smile_by_strike(SMILE, SPOT, T, R1, R2)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.6))
axes[0].plot(by_delta["put_delta"], by_delta["volatility"], color=PRIMARY, linewidth=2)
mark_level(axes[0], 0.5, "ATM"); as_percent(axes[0], decimals=0)
style_axis(axes[0], "Smile in delta space (as quoted)", "Put delta", "Implied volatility",
           "Symmetric-looking. This is the space the market quotes in and the space Malz is defined in.")

axes[1].plot(by_strike["strike"], by_strike["volatility"], color=SECONDARY, linewidth=2)
mark_level(axes[1], forward(SPOT, T, R1, R2), "forward"); as_percent(axes[1], decimals=0)
style_axis(axes[1], "Smile in strike space (as traded)", "Strike (CCY2 per CCY1)", "",
           "The same curve, stretched. Log-normality spreads topside strikes further apart, so the symmetry disappears.")
plt.tight_layout(); plt.show()

Same smile, two views. The delta-space version looks symmetric; the strike-space version does not — because equal steps in delta are *not* equal steps in strike. Log-normality stretches the topside.

### Task E — strike placement, and the six experiments

Practical F asks you to reproduce each of these. Predict each before running it.

In [7]:
base = strike_placement(SMILE, SPOT, T, R1, R2)
print(base.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

 put_delta    label  volatility  strike  pct_from_atm
    0.1000  10% put      0.1288  1.1469      -15.0264
    0.2500  25% put      0.1150  1.2501       -7.3803
    0.5000  50% put      0.1000  1.3497        0.0000
    0.7500 25% call      0.0950  1.4413        6.7866
    0.9000 10% call      0.0968  1.5402       14.1097


> **Does this iterate?** No — and it is worth being explicit, because the relationship *looks* circular: volatility depends on delta, delta depends on strike, strike depends on volatility.
>
> It is not circular here because **Malz is quoted in delta space**. Given a delta, the volatility is known immediately from the quadratic, with no strike involved. Two steps, no solving. Had the smile been parameterised in strike space it genuinely would need a root find — and the reverse direction (`volatility(expiry, strike)` on the surface, below) *does*.

### Experiment 1 — No smile at all

In [8]:
flat = strike_placement(MalzSmile(atm=0.10), SPOT, T, R1, R2)
atm_k = flat.loc[flat.put_delta == 0.50, "strike"].iloc[0]
print(flat[["label", "strike", "pct_from_atm"]].to_string(index=False,
      float_format=lambda v: f"{v:.4f}"))
down = atm_k - flat.loc[flat.put_delta == 0.10, "strike"].iloc[0]
up = flat.loc[flat.put_delta == 0.90, "strike"].iloc[0] - atm_k
print(f"\ndistance ATM -> 10d put:  {down:.4f}")
print(f"distance ATM -> 10d call: {up:.4f}   ({up/down:.3f}x wider)")

   label  strike  pct_from_atm
 10% put  1.1857      -12.1505
 25% put  1.2605       -6.6113
 50% put  1.3497        0.0000
25% call  1.4472        7.2256
10% call  1.5475       14.6555

distance ATM -> 10d put:  0.1640
distance ATM -> 10d call: 0.1978   (1.206x wider)


**Result:** roughly even spacing, with the topside about **21% wider** than the downside (0.1978 versus 0.1640).

That asymmetry is pure log-normality — spot can double but cannot halve twice, so equal probability steps cover more ground upward than downward. Exactly what the book describes: *"roughly equally spaced, with relatively slightly larger differences for topside strikes due to the log-normality of the terminal spot distribution."*

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.6))

for vol, colour in [(0.05, TERTIARY), (0.10, PRIMARY), (0.20, SECONDARY)]:
    p = strike_placement(MalzSmile(atm=vol), SPOT, T, R1, R2)
    axes[0].plot(p["put_delta"], p["pct_from_atm"], "o-", color=colour, label=f"{vol:.0%} vol")

for tenor, colour in [(0.1, TERTIARY), (1.0, PRIMARY), (3.0, SECONDARY)]:
    p = strike_placement(MalzSmile(atm=0.10), SPOT, tenor, R1, R2)
    axes[1].plot(p["put_delta"], p["pct_from_atm"], "o-", color=colour, label=f"T = {tenor}y")

for ax, title, caption in [
    (axes[0], "Lower vol pulls strikes in",
     "A tighter terminal distribution means a given delta sits closer to the ATM."),
    (axes[1], "Shorter tenor pulls strikes in",
     "Same mechanism - both act through sigma x sqrt(T), the width of the distribution."),
]:
    ax.axhline(0, color=MUTED, linewidth=0.8); ax.legend(fontsize=9)
    style_axis(ax, title, "Put delta", "Distance from ATM strike (%)", caption)
plt.tight_layout(); plt.show()

**Result:** identical shapes. Volatility and time enter only through `σ√T`, so halving the volatility does the same thing as quartering the tenor. The same fact from notebook 04, showing up in strike space.

### Experiment 4 — Butterfly and risk reversal

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.6))

for fly, colour in [(0.0, MUTED), (0.005, PRIMARY), (0.020, SECONDARY)]:
    p = strike_placement(MalzSmile(atm=0.10, fly25=fly), SPOT, T, R1, R2)
    axes[0].plot(p["put_delta"], p["pct_from_atm"], "o-", color=colour, label=f"fly {fly:.1%}")

for rr, colour in [(-0.04, TERTIARY), (0.0, MUTED), (0.04, SECONDARY)]:
    p = strike_placement(MalzSmile(atm=0.10, rr25=rr), SPOT, T, R1, R2)
    axes[1].plot(p["put_delta"], p["pct_from_atm"], "o-", color=colour, label=f"RR {rr:+.0%}")

for ax, title, caption in [
    (axes[0], "Higher butterfly pushes strikes out, most in the wings",
     "The fly lifts wing volatility more than the body, so the wing strikes travel furthest."),
    (axes[1], "Risk reversal places strikes asymmetrically",
     "Negative RR makes the downside rich: that strike moves OUT while the topside comes IN."),
]:
    ax.axhline(0, color=MUTED, linewidth=0.8); ax.legend(fontsize=9)
    style_axis(ax, title, "Put delta", "Distance from ATM strike (%)", caption)
plt.tight_layout(); plt.show()

flat_p = strike_placement(MalzSmile(atm=0.10), SPOT, T, R1, R2)
wing_p = strike_placement(MalzSmile(atm=0.10, fly25=0.02), SPOT, T, R1, R2)
for d in (0.25, 0.10):
    a = abs(flat_p.loc[flat_p.put_delta == d, "pct_from_atm"].iloc[0])
    b = abs(wing_p.loc[wing_p.put_delta == d, "pct_from_atm"].iloc[0])
    print(f"  {d:.0%} delta put moved out by {b - a:.2f} percentage points")

  25% delta put moved out by 1.02 percentage points
  10% delta put moved out by 5.00 percentage points


**Result:** the butterfly moves the 10 delta strike **five times** as far as the 25 delta — 5.00 percentage points against 1.02.

The mechanism is direct. The `16·(X − 0.5)²` term is quadratic, so it lifts volatility far more in the wings than near the middle: at the 10 delta the fly contributes `2.56 × fly`, at the 25 delta only `1.00 × fly`. Higher volatility means a wider distribution means the strike must sit further out to keep the same probability of finishing in the money. The book puts it as *"a larger impact at lower delta strikes due to the higher implied volatility"*.

The risk reversal is the asymmetric one. A negative risk reversal makes the **downside** rich, so the downside strike pushes out while the topside pulls in — precisely the book's phrasing: further from the ATM on the rich side, closer on the cheap side.

In [11]:
rows = []
for label, r1, r2 in [("base       r1=2%, r2=5%", 0.02, 0.05),
                      ("high CCY1  r1=10%, r2=5%", 0.10, 0.05),
                      ("low CCY2   r1=2%, r2=0%", 0.02, 0.00),
                      ("high CCY2  r1=2%, r2=12%", 0.02, 0.12)]:
    p = strike_placement(SMILE, SPOT, T, r1, r2)
    row = {"case": label, "forward": forward(SPOT, T, r1, r2)}
    row.update({lbl: k for lbl, k in zip(p["label"], p["strike"])})
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

print(f"\nattainable delta cap at r1=2%:  {max_attainable_put_delta(0.02, 1.0):.4f}")
print(f"attainable delta cap at r1=10%: {max_attainable_put_delta(0.10, 1.0):.4f}")

                    case  forward  10% put  25% put  50% put  25% call  10% call
 base       r1=2%, r2=5%   1.3396   1.1469   1.2501   1.3497    1.4413    1.5402
high CCY1  r1=10%, r2=5%   1.2366   1.0651   1.1626   1.2593    1.3595    1.5906
 low CCY2   r1=2%, r2=0%   1.2743   1.0910   1.1891   1.2839    1.3710    1.4650
high CCY2  r1=2%, r2=12%   1.4367   1.2301   1.3407   1.4476    1.5458    1.6518

attainable delta cap at r1=2%:  0.9802
attainable delta cap at r1=10%: 0.9048


**Result:** the claim holds for four of the five strikes and **reverses for the 10 delta call.**

That is not a bug and it is worth understanding. A put delta can never exceed `exp(-r1·T)` in absolute terms, because `Δput = exp(-r1·T)·[N(d1) - 1]` and the bracket lives in `(-1, 0)`. At a 10% CCY1 rate over a year that ceiling is 0.9048 — and the 90% delta strike we are solving for sits *just* beneath it.

As the rate rises, `exp(r1·T)·Δ + 1` collapses toward zero, the inverse normal dives, and the strike is forced **out** faster than the lower forward pulls it **in**. The two effects fight and near the ceiling the delta-pinning wins.

The book's statement is about the forward moving, and it is right for the body of the smile. It quietly assumes deltas comfortably inside the attainable range — which the book's own modest-rate examples always are. Both behaviours are asserted in `tests/test_smile.py`.

This is also why `strike_placement` **omits** unattainable deltas rather than raising: at a 20% CCY1 rate over two years the cap is 0.67, and the 75 and 90 delta strikes genuinely do not exist.

In [12]:
print("At r1 = 20%, T = 2y:")
print(f"  attainable cap: {max_attainable_put_delta(0.20, 2.0):.4f}")
print(strike_placement(MalzSmile(atm=0.10), SPOT, 2.0, 0.20, 0.05).to_string(
    index=False, float_format=lambda v: f"{v:.4f}"))
print("\n75% and 90% put deltas are simply absent - they do not exist at this rate.")

At r1 = 20%, T = 2y:
  attainable cap: 0.6703
 put_delta   label  volatility  strike  pct_from_atm
    0.1000 10% put      0.1000  0.8397      -21.3881
    0.2500 25% put      0.1000  0.9292      -13.0121
    0.5000 50% put      0.1000  1.0682        0.0000

75% and 90% put deltas are simply absent - they do not exist at this rate.


---

# The assembled surface

The book stops here. It has built an ATM curve (Practical E) and a smile (Practical F) and never joins them.

Joining them is the point of the whole first half:

```
tenor dates (Practical D)
    → ATM curve with day weights (Practical E)
        → Malz smile per tenor (Practical F)
            → vol(expiry_date, strike)
```

That last line is what a desk needs. Chapter 7: to quote a consistent price for **any** expiry date and **any** strike — including contracts nobody has quoted before — you need a surface. That flexibility is what makes an OTC market work.

In [13]:
HORIZON = date(2014, 6, 11)
surface = example_surface(HORIZON)

print("The run of market instruments, as a desk would see it:\n")
print(surface.grid().to_string(index=False, float_format=lambda v: f"{v:.4f}"))

The run of market instruments, as a desk would see it:

tenor     expiry  years   10%P   25%P   50%P   25%C   10%C
   1W 2014-06-18 0.0192 0.0743 0.0711 0.0685 0.0696 0.0719
   2W 2014-06-25 0.0384 0.0767 0.0730 0.0700 0.0710 0.0735
   1M 2014-07-10 0.0795 0.0799 0.0756 0.0720 0.0728 0.0754
   2M 2014-08-11 0.1671 0.0837 0.0788 0.0745 0.0752 0.0781
   3M 2014-09-11 0.2521 0.0870 0.0814 0.0765 0.0772 0.0803
   6M 2014-12-11 0.5014 0.0926 0.0859 0.0800 0.0804 0.0838
   9M 2015-03-11 0.7479 0.0961 0.0887 0.0820 0.0823 0.0858
   1Y 2015-06-11 1.0000 0.0988 0.0908 0.0835 0.0838 0.0876
   2Y 2016-06-09 1.9973 0.1051 0.0956 0.0870 0.0871 0.0915


In [14]:
expiry = surface.expiries[5]   # the 6M tenor
print(f"Pricing arbitrary strikes at the {surface.tenor_smiles[5].tenor} expiry ({expiry}):\n")
for strike in [1.15, 1.25, surface.strike_for_delta(expiry, 0.50), 1.45, 1.55]:
    print(f"  K = {strike:.4f}  ->  {surface.volatility(expiry, strike):.4%}")

mid = surface.expiries[4] + (surface.expiries[5] - surface.expiries[4]) / 2
print(f"\nAnd a date no tenor quotes ({mid}):")
print(f"  ATM = {surface.atm(mid):.4%}, K = 1.35 -> {surface.volatility(mid, 1.35):.4%}")

Pricing arbitrary strikes at the 6M expiry (2014-12-11):

  K = 1.1500  ->  9.6812%
  K = 1.2500  ->  8.7722%
  K = 1.3154  ->  8.0000%
  K = 1.4500  ->  8.5219%
  K = 1.5500  ->  8.7045%

And a date no tenor quotes (2014-10-26):
  ATM = 7.8829%, K = 1.35 -> 7.9071%


> **This direction *does* iterate.** Going from delta to strike is two steps. Going from **strike** to volatility is circular — the smile is quoted in delta, so you need the delta, which needs a volatility. It is solved by fixed-point iteration from the ATM and converges in a handful of steps because the smile is shallow in delta. `tests/test_surface.py` asserts the fixed point is self-consistent, not merely that it terminates.

### The surface, in three dimensions

In [15]:
mesh = surface.surface_mesh(points=30)
pivot = mesh.pivot_table(index="years", columns="put_delta", values="volatility")

fig = go.Figure(go.Surface(
    x=pivot.columns.to_numpy(), y=pivot.index.to_numpy(), z=pivot.to_numpy(),
    colorscale="Blues", showscale=False,
    contours={"z": {"show": True, "usecolormap": True}},
))
fig.update_layout(
    scene=dict(xaxis_title="Put delta", yaxis_title="Time to expiry (years)",
               zaxis_title="Implied volatility"),
    height=560,
    title="The volatility surface — ATM curve along one axis, smile along the other",
)
fig.show()

Rotate it. Two structures are visible and they are the two halves of Part II:

- Along the **time** axis: the ATM curve, sloping up. That is Chapter 11.
- Along the **delta** axis: the smile at each tenor, tilted and lifted. That is Chapter 12.

The surface is nothing more than those two things crossed. Which is exactly why the book teaches them separately — and exactly why joining them yourself is worth doing.

### Checking it is arbitrage-free

In [16]:
check = surface.check_no_calendar_arbitrage()
print(check.to_string(index=False, float_format=lambda v: f"{v:.6f}"))
print(f"\nany negative forward variance? {check['negative'].any()}")

tenor    years      atm  total_variance  forward_variance  negative
   1W 0.019178 0.068500        0.000090               NaN     False
   2W 0.038356 0.070000        0.000188          0.000098     False
   1M 0.079452 0.072000        0.000412          0.000224     False
   2M 0.167123 0.074500        0.000928          0.000516     False
   3M 0.252055 0.076500        0.001475          0.000548     False
   6M 0.501370 0.080000        0.003209          0.001734     False
   9M 0.747945 0.082000        0.005029          0.001820     False
   1Y 1.000000 0.083500        0.006972          0.001943     False
   2Y 1.997260 0.087000        0.015117          0.008145     False

any negative forward variance? False


### And a deliberately broken one, to show the check earns its place

In [17]:
broken = VolatilitySurface(
    HORIZON, 1.30, 0.0, 0.0,
    [TenorSmile("6M", 0.20, 0.0, 0.0),
     TenorSmile("1Y", 0.20, 0.0, 0.0),
     TenorSmile("2Y", 0.10, 0.0, 0.0)],
)
bad = broken.check_no_calendar_arbitrage()
print(bad.to_string(index=False, float_format=lambda v: f"{v:.6f}"))
print(f"\nany negative forward variance? {bad['negative'].any()}   <- caught")
print("\nThe 2yr is quoted BELOW the 1yr by enough that total variance falls.")
print("Sell the 1yr, buy the 2yr: you pay less for more total uncertainty.")

tenor    years      atm  total_variance  forward_variance  negative
   6M 0.501370 0.200000        0.020055               NaN     False
   1Y 1.000000 0.200000        0.040000          0.019945     False
   2Y 1.997260 0.100000        0.019973         -0.020027      True

any negative forward variance? True   <- caught

The 2yr is quoted BELOW the 1yr by enough that total variance falls.
Sell the 1yr, buy the 2yr: you pay less for more total uncertainty.


### What this surface is ignoring

A surface that does not tell you what it simplifies is worse than no surface. This one says so in code:

In [18]:
print(surface.explain_simplifications())

This surface simplifies in the following ways:

1. Malz smile on OUTRIGHT deltas, not the broker fly the interbank market actually trades. Chapter 12 spends several pages on why those differ: broker fly strikes are generated ignoring the risk reversal, so they are not the outright 25 delta strikes, and a broker fly carries vanna when valued on the smile. This is the single largest simplification in the surface.
2. The Malz form implies a 25d/10d risk reversal multiplier of exactly 1.6. Chapter 12 notes the market value is usually nearer 1.8, so the 10 delta skew is systematically understated.
3. Smile parameters are interpolated linearly in time between tenors. Real desks differ on whether to interpolate in delta space, strike space or model-parameter space - Chapter 12 says so explicitly and does not pick one.
4. Spot delta throughout. No forward-delta convention (Chapter 12 notes long-dated G10 and EM risk reversals are usually quoted on forward delta) and no premium-adjusted delta f

The first one is the big one and deserves restating in full.

**The interbank market does not trade the butterfly this model uses.** It trades the **broker fly**, whose strikes are generated at `ATM + fly` volatility — *ignoring the risk reversal entirely*. So broker fly strikes are not the outright 25 delta strikes, and the two instruments differ materially in skewed or long-dated pairs.

Chapter 12's worked example: in a 5-year AUD/JPY, the outright 25 delta put strike is 46.05 while the broker fly's is 49.60. Not a rounding difference. And because of that strike placement, a broker fly carries **vanna** when valued on the smile — which is why Chapter 12 shows AUD/JPY 25 delta flies going *more negative* at longer tenors even as the ATM and risk reversal both rise, a result that looks backwards until you know where the strikes are.

Practical F builds the outright-delta smile. So does this. Both are useful; neither is what the broker quotes.

## Experiments on the surface

### Experiment 6 — Does the smile survive the round trip?

**Predict:** solve for the strike at a delta, then ask the surface for that strike's volatility. Must it come back to the smile value?

In [19]:
expiry = surface.expiries[7]
print(f"{surface.tenor_smiles[7].tenor} expiry, {expiry}\n")
print(f"{'delta':>8} {'strike':>10} {'smile vol':>11} {'surface vol':>13} {'error':>10}")
for d in (0.10, 0.25, 0.50, 0.75, 0.90):
    k = surface.strike_for_delta(expiry, d)
    smile_v = float(surface.smile_at(expiry).volatility(d))
    surf_v = surface.volatility(expiry, k)
    print(f"{d:>8.2f} {k:>10.4f} {smile_v:>11.4%} {surf_v:>13.4%} {abs(surf_v-smile_v):>10.2e}")

1Y expiry, 2015-06-11

   delta     strike   smile vol   surface vol      error
    0.10     1.1745     9.8828%       9.8828%   5.45e-14
    0.25     1.2531     9.0800%       9.0800%   2.46e-14
    0.50     1.3316     8.3500%       8.3500%   0.00e+00
    0.75     1.4097     8.3800%       8.3800%   7.52e-15
    0.90     1.4930     8.7628%       8.7628%   1.92e-14


**Result:** exact. It has to be — the iteration is solving for precisely the self-consistency this checks. If it were not exact the fixed point would be wrong, and a round-trip test is the cleanest way to catch that.

### Experiment 7 — Attach day weights and watch the surface saw-tooth

**Predict:** hand the surface a weighted ATM curve with the weekend at zero. What happens between tenors?

In [20]:
from datetime import timedelta
from fxds.atm_curve import WeightedATMCurve, WEEKEND_ZERO_WEIGHTS

weighted = VolatilitySurface(
    horizon=HORIZON, spot=surface.spot, r_ccy1=surface.r_ccy1, r_ccy2=surface.r_ccy2,
    tenor_smiles=surface.tenor_smiles,
    weights=WeightedATMCurve(HORIZON, 0.08, weekday_weights=dict(WEEKEND_ZERO_WEIGHTS)),
)

start = surface.expiries[1]
days = [start + timedelta(days=i) for i in range(56)]
plain_vols = [surface.atm(d) for d in days]
saw_vols = [weighted.atm(d) for d in days]

fig, ax = plt.subplots(figsize=(12, 4.4))
ax.plot(days, plain_vols, color=MUTED, linewidth=1.8, label="interpolated only")
ax.plot(days, saw_vols, color=PRIMARY, linewidth=1.6, label="with day weights")
ax.legend(); as_percent(ax, decimals=1); ax.tick_params(axis="x", rotation=30)
style_axis(ax, "The ATM curve inside a surface, with and without day weights",
           "Expiry date", "ATM implied volatility",
           "The grey line is the smooth interpolation between quoted tenors. The blue line adds the weekend effect from Practical E - the surface now saw-tooths day to day, as a real one does.")
plt.tight_layout(); plt.show()

print(f"smooth curve, day-to-day std of changes: {np.std(np.diff(plain_vols)):.2e}")
print(f"weighted curve:                          {np.std(np.diff(saw_vols)):.2e}")

smooth curve, day-to-day std of changes: 4.72e-05
weighted curve:                          6.23e-04


**Result:** the surface inherits the saw-tooth. That is the join working — Practical E's day weights reaching all the way through to a quoted volatility for an arbitrary strike and date.

Note the composition here is a **choice**, not a market convention. Chapter 11 says desks combine a core curve with weights on top; it does not say how. This one scales the interpolated ATM by the weighted-to-flat ratio. It is documented in the module and in `notes/deviations.md`, and a different desk would do it differently.

## Common misconceptions

**"A 25 delta put has -25% delta, so pass -0.25 to the Malz formula."**
No — Malz takes the **positive quoted** delta (0.25). The Black-Scholes strike functions take the **signed** value (-0.25). Two conventions, three lines apart. Every docstring in `fxds/smile.py` says which it wants.

**"Smile and skew mean the same thing."**
The **smile** is the whole shape. The **skew** is its tilt — what the risk reversal measures. A symmetric smile has wings but no skew.

**"A positive risk reversal means the market is bullish."**
It means topside strikes are richer *in volatility terms*. Chapter 12: higher volatility sits on the **weaker** side of spot — the direction spot is more likely to jump. USD/EM pairs have topside-rich smiles because the tail risk is EM devaluation, which is a statement about tails, not direction.

**"The butterfly quoted in the broker market is the one this model uses."**
It is not. The broker fly's strikes are generated ignoring the risk reversal, so they differ from the outright strikes — by 3.5 big figures in Chapter 12's 5-year AUD/JPY example. The single largest simplification in this surface.

**"Malz gives 10 delta risk reversals for free."**
It gives you `1.6 × RR25`. The market is nearer 1.8. Fine for intuition, not for quoting.

**"You can always solve for a 90 delta put."**
Only if `exp(-r1·T) > 0.90`. At a 10% CCY1 rate over a year the ceiling is 0.9048; at 20% over two years it is 0.67 and the strike does not exist. Experiment 5.

**"A volatility surface is a model of reality."**
It is a **quoting convention** that interpolates observed prices. Chapter 12 closes on exactly this: *"there is nothing about the FX derivatives market that makes the ATM, risk reversal, and butterfly approach the only possible way of representing the volatility smile."*

## Check yourself

1. The 25 delta call is at 9.5% and the 25 delta put at 11.5%, with the ATM at 10%. What are the risk reversal and the butterfly?
2. Why do the 2 and the 16 appear in the Malz formula?
3. You get a strike above the forward for a 25 delta *put*. What did you do wrong?
4. A negative risk reversal — which strike moves further from the ATM, and why?
5. Your surface prices a 6-month at 8% and a 1-year at 7%. Is that allowed?

In [21]:
#@title Answers — run this cell to reveal
from IPython.display import Markdown
Markdown(r'''
**1.** `RR = 9.5% − 11.5% = **−2%**` (downside rich — the EUR/USD shape). `fly = (9.5% + 11.5%)/2 − 10% = **+0.5%**` (wings above the middle). The risk reversal is the difference; the butterfly is the average minus the ATM.

**2.** They are exactly what makes the quadratic pass through the quoted points. At `X = 0.25`, `(X − 0.5) = −0.25`, so the RR coefficient of 2 gives `2 × (−0.25) = −0.5` — the half in `σ_ATM + fly − ½RR`. The fly coefficient of 16 gives `16 × 0.0625 = 1`, the unit weight on the butterfly. Not fitted parameters; forced by the definitions.

**3.** You passed the **positive** quoted delta instead of the signed one. `strike_from_put_delta` wants `−0.25` for a 25 delta put. This implementation raises rather than returning the wrong strike silently, which is what the book's version would do.

**4.** The **downside** strike moves further out. A negative risk reversal makes downside strikes richer in volatility, and higher volatility means a wider distribution, so the strike must sit further away to keep the same probability of finishing in the money. The topside, now cheaper, pulls in. Chapter 12: further out on the rich side, closer on the cheap side.

**5.** **No** — check the variance. `var(6m) = 0.08² × 0.5 = 0.0032`; `var(1y) = 0.07² × 1.0 = 0.0049`. Variance rises, so it *is* allowed, despite the volatility falling. This is exactly why curves are checked in variance and not volatility: a downward-sloping volatility curve is perfectly normal (Chapter 7 calls it inverted, typical of stressed markets). Only *falling variance* is an arbitrage.
''')


**1.** `RR = 9.5% − 11.5% = **−2%**` (downside rich — the EUR/USD shape). `fly = (9.5% + 11.5%)/2 − 10% = **+0.5%**` (wings above the middle). The risk reversal is the difference; the butterfly is the average minus the ATM.

**2.** They are exactly what makes the quadratic pass through the quoted points. At `X = 0.25`, `(X − 0.5) = −0.25`, so the RR coefficient of 2 gives `2 × (−0.25) = −0.5` — the half in `σ_ATM + fly − ½RR`. The fly coefficient of 16 gives `16 × 0.0625 = 1`, the unit weight on the butterfly. Not fitted parameters; forced by the definitions.

**3.** You passed the **positive** quoted delta instead of the signed one. `strike_from_put_delta` wants `−0.25` for a 25 delta put. This implementation raises rather than returning the wrong strike silently, which is what the book's version would do.

**4.** The **downside** strike moves further out. A negative risk reversal makes downside strikes richer in volatility, and higher volatility means a wider distribution, so the strike must sit further away to keep the same probability of finishing in the money. The topside, now cheaper, pulls in. Chapter 12: further out on the rich side, closer on the cheap side.

**5.** **No** — check the variance. `var(6m) = 0.08² × 0.5 = 0.0032`; `var(1y) = 0.07² × 1.0 = 0.0049`. Variance rises, so it *is* allowed, despite the volatility falling. This is exactly why curves are checked in variance and not volatility: a downward-sloping volatility curve is perfectly normal (Chapter 7 calls it inverted, typical of stressed markets). Only *falling variance* is an arbitrage.


## Where next

You now have the whole first half of the book working end to end: dates, an ATM curve, a smile, and a surface that joins them.

What you do **not** have, and should know you do not have:

- The **broker fly** (Chapter 12). The largest gap between this surface and a traded one.
- **Premium-adjusted delta** and **forward delta** conventions (Chapters 8 and 14).
- **Adapted vega**, **weighted vega**, **rega** and **sega** — the position-level smile greeks (Chapters 12 and 14).
- Anything about **credit** or real **interest rate curves** — set aside by the book itself, in its Preface.

Chapter 13 and Practical G build a probability density function from option prices, which is the natural continuation and is out of scope for this pass. The surface in `fxds/surface.py` is the thing you would extend.